In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [2]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score
)
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
# Load the Kaggle competition training data
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
train_df = train_df.drop(columns=['id'])

# Separate features and target
X = train_df.drop(columns=['Irrigation_Need'])
y = train_df['Irrigation_Need']

# Identify categorical columns
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns: {cat_cols}")
print(f"Numeric columns count: {len(num_cols)}")

# Encode categorical columns
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le

# Fill any missing values in numeric columns with median
for col in num_cols:
    if X[col].isnull().sum() > 0:
        X[col] = X[col].fillna(X[col].median())

# Encode the target variable
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)
class_names = le_target.classes_

print(f"\nEncoded class labels: {list(class_names)}")
print(f"Class index mapping: { {name: idx for idx, name in enumerate(class_names)} }")
print("\nPreprocessing complete.")

print(f"Training data shape: {train_df.shape}")
print(f"\nTarget class distribution:")
print(train_df['Irrigation_Need'].value_counts())
print(f"\nClass proportions:")
print(train_df['Irrigation_Need'].value_counts(normalize=True).round(4))

Categorical columns: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
Numeric columns count: 11

Encoded class labels: ['High', 'Low', 'Medium']
Class index mapping: {'High': 0, 'Low': 1, 'Medium': 2}

Preprocessing complete.
Training data shape: (630000, 20)

Target class distribution:
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64

Class proportions:
Irrigation_Need
Low       0.5872
Medium    0.3795
High      0.0333
Name: proportion, dtype: float64


In [4]:

# Separate features and target
X = train_df.drop(columns=['Irrigation_Need'])
y = train_df['Irrigation_Need']

# Identify categorical columns
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns: {cat_cols}")
print(f"Numeric columns count: {len(num_cols)}")

# Encode categorical columns
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le

# Fill any missing values in numeric columns with median
for col in num_cols:
    if X[col].isnull().sum() > 0:
        X[col] = X[col].fillna(X[col].median())

# Encode the target variable
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)
class_names = le_target.classes_

print(f"\nEncoded class labels: {list(class_names)}")
print(f"Class index mapping: { {name: idx for idx, name in enumerate(class_names)} }")
print("\nPreprocessing complete.")

Categorical columns: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
Numeric columns count: 11

Encoded class labels: ['High', 'Low', 'Medium']
Class index mapping: {'High': 0, 'Low': 1, 'Medium': 2}

Preprocessing complete.


In [5]:
# Split using only the Kaggle train data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"Training set size:  {X_train.shape[0]:,}")
print(f"Test set size:      {X_test.shape[0]:,}")

Training set size:  504,000
Test set size:      126,000


In [6]:
# Train the Gaussian Naive Bayes model
gnb = GaussianNB()
gnb.fit(X_train, y_train)

print("Gaussian Naive Bayes model trained.")
print(f"Classes learned: {list(class_names)}")

Gaussian Naive Bayes model trained.
Classes learned: ['High', 'Low', 'Medium']


In [7]:
# Generate predicted probabilities for all three classes
y_proba = gnb.predict_proba(X_test)

# Display probability columns and a sample
proba_df = pd.DataFrame(y_proba, columns=[f'P({cls})' for cls in class_names])

print("Predicted probability sample (first 10 rows):")
print(proba_df.head(10).round(4))
print(f"\nProbability shape: {y_proba.shape}")

Predicted probability sample (first 10 rows):
   P(High)  P(Low)  P(Medium)
0   0.0000  0.9354     0.0646
1   0.0007  0.2978     0.7015
2   0.0000  0.5390     0.4610
3   0.0000  0.8841     0.1159
4   0.0000  0.7500     0.2500
5   0.1176  0.2221     0.6603
6   0.0042  0.1684     0.8274
7   0.0168  0.2504     0.7328
8   0.0000  0.3521     0.6479
9   0.0075  0.3497     0.6428

Probability shape: (126000, 3)


In [8]:
# Baseline: assign whichever class has the highest predicted probability
y_pred_baseline = gnb.predict(X_test)

# Baseline metrics
baseline_bal_acc = balanced_accuracy_score(y_test, y_pred_baseline)

print("=== Baseline Predictions (Default Rule) ===")
print(f"Balanced Accuracy: {baseline_bal_acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_baseline, target_names=class_names))

=== Baseline Predictions (Default Rule) ===
Balanced Accuracy: 0.6911

Classification Report:
              precision    recall  f1-score   support

        High       0.72      0.42      0.53      4202
         Low       0.85      0.90      0.88     73983
      Medium       0.79      0.75      0.77     47815

    accuracy                           0.83    126000
   macro avg       0.79      0.69      0.73    126000
weighted avg       0.82      0.83      0.82    126000



In [9]:
# Find the index of 'High' in the encoded class array
high_class_index = list(class_names).index('High')
print(f"'High' class is at index: {high_class_index}")

# Extract the predicted probability for 'High' for each test row
p_high = y_proba[:, high_class_index]

print(f"\nSummary of P(High) probabilities:")
print(pd.Series(p_high).describe().round(4))

'High' class is at index: 0

Summary of P(High) probabilities:
count    126000.0000
mean          0.0329
std           0.1118
min           0.0000
25%           0.0000
50%           0.0000
75%           0.0045
max           0.9389
dtype: float64


In [10]:
# Scan a range of thresholds to observe how balanced accuracy changes
thresholds = np.arange(0.05, 0.55, 0.05)

print(f"{'Threshold':<12} {'Balanced Accuracy':<20} {'High Recall':<15} {'High Precision':<15}")
print("-" * 62)

for t in thresholds:
    # Apply threshold: if P(High) >= t, predict High; otherwise use argmax of remaining probs
    y_pred_t = np.where(p_high >= t, high_class_index, np.argmax(
        np.column_stack([
            y_proba[:, i] for i in range(y_proba.shape[1]) if i != high_class_index
        ]), axis=1
    ).copy())

    # Remap non-High indices (0,1 from the 2-col array) back to original class indices
    other_indices = [i for i in range(y_proba.shape[1]) if i != high_class_index]
    y_pred_mapped = np.where(
        p_high >= t,
        high_class_index,
        np.array([other_indices[i] for i in np.argmax(
            y_proba[:, other_indices], axis=1
        )])
    )

    bal_acc = balanced_accuracy_score(y_test, y_pred_mapped)

    # Pull High recall and precision from classification report dict
    from sklearn.metrics import precision_recall_fscore_support
    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred_mapped, labels=[high_class_index], zero_division=0)

    print(f"{t:<12.2f} {bal_acc:<20.4f} {r[0]:<15.4f} {p[0]:<15.4f}")

Threshold    Balanced Accuracy    High Recall     High Precision 
--------------------------------------------------------------
0.05         0.7930               0.9050          0.2721         
0.10         0.8041               0.8703          0.3635         
0.15         0.8035               0.8365          0.4305         
0.20         0.7966               0.7949          0.4880         
0.25         0.7880               0.7523          0.5485         
0.30         0.7705               0.6885          0.5878         
0.35         0.7510               0.6202          0.6293         
0.40         0.7286               0.5450          0.6669         
0.45         0.7069               0.4733          0.7093         
0.50         0.6861               0.4060          0.7476         


In [11]:
# Apply chosen threshold of 0.15 for final evaluation
THRESHOLD = 0.15

other_indices = [i for i in range(y_proba.shape[1]) if i != high_class_index]

y_pred_threshold = np.where(
    p_high >= THRESHOLD,
    high_class_index,
    np.array([other_indices[i] for i in np.argmax(
        y_proba[:, other_indices], axis=1
    )])
)

threshold_bal_acc = balanced_accuracy_score(y_test, y_pred_threshold)

print(f"=== Threshold Predictions (threshold = {THRESHOLD}) ===")
print(f"Balanced Accuracy: {threshold_bal_acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_threshold, target_names=class_names))

=== Threshold Predictions (threshold = 0.15) ===
Balanced Accuracy: 0.8035

Classification Report:
              precision    recall  f1-score   support

        High       0.43      0.84      0.57      4202
         Low       0.85      0.90      0.88     73983
      Medium       0.81      0.67      0.73     47815

    accuracy                           0.81    126000
   macro avg       0.70      0.80      0.73    126000
weighted avg       0.82      0.81      0.81    126000



In [12]:
from sklearn.metrics import precision_recall_fscore_support

# Baseline High stats
p_b, r_b, f_b, _ = precision_recall_fscore_support(
    y_test, y_pred_baseline, labels=[high_class_index], zero_division=0
)

# Threshold High stats
p_t, r_t, f_t, _ = precision_recall_fscore_support(
    y_test, y_pred_threshold, labels=[high_class_index], zero_division=0
)

summary = pd.DataFrame({
    'Approach': ['Baseline (default rule)', f'Threshold = {THRESHOLD}'],
    'Balanced Accuracy': [
        round(balanced_accuracy_score(y_test, y_pred_baseline), 4),
        round(balanced_accuracy_score(y_test, y_pred_threshold), 4)
    ],
    'High - Precision': [round(p_b[0], 4), round(p_t[0], 4)],
    'High - Recall':    [round(r_b[0], 4), round(r_t[0], 4)],
    'High - F1':        [round(f_b[0], 4), round(f_t[0], 4)]
})

print("=== Baseline vs. Threshold Comparison ===")
print(summary.to_string(index=False))

=== Baseline vs. Threshold Comparison ===
               Approach  Balanced Accuracy  High - Precision  High - Recall  High - F1
Baseline (default rule)             0.6911            0.7207         0.4231     0.5332
       Threshold = 0.15             0.8035            0.4305         0.8365     0.5684


## Discussion

### Class Selected
I focused on the **'High'** irrigation need class. This class is the rarest in the dataset, only about 3.3% of all rows. It is also the most important to catch correctly, because missing a 'High' prediction means a farmer gets no warning when their crops urgently need water.

### Metric Selected
I used **balanced accuracy** as my main metric. This metric treats all three classes equally, which matters here because the classes are very uneven in size. I also looked at **recall** and **precision** for the 'High' class to understand the tradeoff from changing the threshold.

### Threshold Chosen
By default, the model predicts 'High' only when it is at least 50% confident. I lowered that bar to **0.15**, meaning if the model thinks there is at least a 15% chance the answer is 'High', it predicts 'High'. I tested thresholds from 0.05 to 0.50 and found that 0.15 gave a good balance between catching more 'High' cases and not over-predicting them.

### How the Metric Changed
| | Baseline | Threshold = 0.15 |
|---|---|---|
| Balanced Accuracy | 0.6911 | 0.8035 |
| High — Recall | 0.42 | 0.84 |
| High — Precision | 0.72 | 0.43 |
| High — F1 | 0.53 | 0.57 |

Balanced accuracy improved by about **+0.11**, and the model now correctly identifies **twice as many** true 'High' cases.

### Tradeoff Observed
By lowering the threshold, the model catches more 'High' cases (recall goes up), but it also makes more mistakes by labeling some 'Low' or 'Medium' cases as 'High' (precision goes down). In this situation, that tradeoff makes sense, it is better to give an extra warning than to miss a farmer who really needs irrigation.

### Naive Bayes vs. Prior Models
Naive Bayes is a much simpler model than the ones used previously. It assumes all features are completely independent of each other, which is not realistic here — temperature, wind speed, and soil moisture all influence each other in the real world. Because of this, Naive Bayes scores noticeably lower:

| Model | CV Balanced Accuracy |
|---|---|
| Gaussian NB | 0.8035 (test set) |
| Random Forest | 0.9553 |
| XGBoost | 0.9620 |
| CatBoost | 0.9690 |
| LightGBM | 0.9702 |

The main advantage of Naive Bayes is that it gives you probability estimates for each class, which made the threshold analysis in this assignment possible. The ensemble models are much more accurate overall, but Naive Bayes is a useful starting point for understanding how probability thresholds work.